In [1]:
import json
import networkx as nx



In [2]:

def load_graph(json_path: str) -> nx.DiGraph:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    G = nx.DiGraph()

    for func_id, meta in data.items():
        G.add_node(func_id, **meta)
        for callee in meta["calls"]:
            G.add_edge(func_id, callee)

    return G


def find_node_by_name(G: nx.DiGraph, function_name: str):
    """
    Returns all nodes matching a function name (since names are not unique)
    """
    return [
        n for n, attr in G.nodes(data=True)
        if attr.get("name") == function_name
    ]


def get_weakly_connected_component(G: nx.DiGraph, node: str):
    """
    Returns the subgraph containing the node (ignores edge direction)
    """
    for component in nx.weakly_connected_components(G):
        if node in component:
            return G.subgraph(component).copy()

    return None


def cytoscape_workflow_largest_component(json_path: str, function_name: str):
    """
    1. Load graph
    2. Find all nodes matching function_name
    3. Extract their connected components
    4. Return node metadata for the largest one
    """
    G = load_graph(json_path)

    matching_nodes = find_node_by_name(G, function_name)

    if not matching_nodes:
        raise ValueError(f"No function found with name: {function_name}")

    components = []

    for node in matching_nodes:
        subgraph = get_weakly_connected_component(G, node)
        if subgraph:
            components.append(subgraph)

    if not components:
        return {}

    # choose largest component
    largest = max(components, key=lambda g: g.number_of_nodes())

    # extract node data
    result = {
        node: largest.nodes[node]
        for node in largest.nodes
    }

    return result




In [3]:
json_path = "output.json"
function_name = "cytoscape_workflow_largest_component"

result = cytoscape_workflow_largest_component(json_path, function_name)

print(f"Nodes in component: {len(result)}")
for k, v in list(result.items())[:5]:  # preview
    print(k, "->", v)

Nodes in component: 28
cytoscape_main.py::cytoscape_workflow_by_tag -> {'name': 'cytoscape_workflow_by_tag', 'file': 'cytoscape_main.py', 'docstring': 'Complete workflow: filter by tag and create Cytoscape visualization.\n\nArgs:\n    data_path: Path to JSON data file\n    tag: Tag to filter by\n    port: Port to run the server on\n    remove_isolated: Remove isolated nodes', 'calls': ['data_loader.py::load_data', 'data_loader.py::filter_by_tag', 'graph_builder.py::build_graph', 'network_stats.py::print_statistics', 'cytoscape_visualizer.py::run_cytoscape_visualization']}
data_loader.py::load_data -> {'name': 'load_data', 'file': 'data_loader.py', 'docstring': 'Load JSON data from file.\n\nArgs:\n    filepath: Path to the JSON file\n    \nReturns:\n    List of node dictionaries', 'calls': []}
data_loader.py::filter_by_tag -> {'name': 'filter_by_tag', 'file': 'data_loader.py', 'docstring': 'Filter nodes that have the specified tag.\n\nArgs:\n    data: List of node dictionaries\n    tag:

In [ ]:
Yes. In a directed graph this is a standard “predecessor” query: you are asking for all nodes with an incoming edge to a given node.

Given a key like:

```
cytoscape_main.py::cytoscape_workflow_by_titles
```

you want all nodes that *call into it*.

---

## Using NetworkX

Assuming you already built:

```python
G = nx.DiGraph()
```

You can directly query incoming edges:

### 1. Immediate callers (direct parents)

```python id="p8k2l1"
callers = list(G.predecessors(
    "cytoscape_main.py::cytoscape_workflow_by_titles"
))
```

This returns:

* all functions that directly call it

---

### 2. Full upstream dependency chain

If you want everything that eventually leads into it:

```python id="x2m9ab"
all_callers = nx.ancestors(
    G,
    "cytoscape_main.py::cytoscape_workflow_by_titles"
)
```

This returns:

* transitive closure of incoming edges
* i.e. “everything that can reach this node”

---

## Conceptually

Let:

* $G = (V, E)$ be your function graph
* $E = (f_i \rightarrow f_j)$ means “$f_i$ calls $f_j$”

Then:

### Direct callers:

$$
{ f_i \mid (f_i, f_j) \in E }
$$

### All callers:

$$
{ f_i \mid f_i \leadsto f_j }
$$

---

## Practical wrapper (recommended)

Add this to `grapher.py`:

```python id="v9kq1p"
def get_callers(G, function_id: str, transitive: bool = False):
    if transitive:
        return nx.ancestors(G, function_id)
    else:
        return set(G.predecessors(function_id))
```

Usage:

```python id="c1n9zd"
get_callers(G, "cytoscape_main.py::cytoscape_workflow_by_titles")
```

---

## Common confusion point

Your graph direction matters:

* edge: `A -> B` means A calls B
* so:

  * `successors(A)` = functions A calls
  * `predecessors(B)` = functions that call B

---

## Optional improvement

If you later want debugging clarity, you can also label edges:

```python id="z8p3kf"
G.add_edge(func_id, callee, type="calls")
```

This becomes useful if you extend beyond function calls later.

---

If you want, next step is a “bi-directional query tool” so you can ask:

* “what does this function depend on?”
* “what depends on this function?”
* “what is the full impact radius?”

in a single consistent API.


In [5]:
import json
import networkx as nx

with open("output.json", "r", encoding="utf-8") as f:
    data = json.load(f)

G = nx.DiGraph()

for func_id, meta in data.items():
    # add node (optional but useful for metadata queries)
    G.add_node(func_id, **meta)

    # add edges: caller -> callee
    for callee in meta["calls"]:
        G.add_edge(func_id, callee)

In [9]:

#this returns:* all functions that directly call it

target = "data_loader.py::load_data"


callers = list(G.predecessors(target))

print(callers)
for caller in callers:
    print(caller)


['cytoscape_main.py::cytoscape_workflow_by_tag', 'cytoscape_main.py::cytoscape_workflow_by_titles', 'cytoscape_main.py::cytoscape_workflow_largest_component', 'cytoscape_main.py::cytoscape_workflow_by_category', 'pyvis_main.py::workflow_by_tag', 'pyvis_main.py::workflow_by_titles', 'pyvis_main.py::workflow_largest_component', 'pyvis_main.py::workflow_by_category']
cytoscape_main.py::cytoscape_workflow_by_tag
cytoscape_main.py::cytoscape_workflow_by_titles
cytoscape_main.py::cytoscape_workflow_largest_component
cytoscape_main.py::cytoscape_workflow_by_category
pyvis_main.py::workflow_by_tag
pyvis_main.py::workflow_by_titles
pyvis_main.py::workflow_largest_component
pyvis_main.py::workflow_by_category


In [10]:


# If you want everything that eventually leads into it:

all_callers = nx.ancestors(
    G,
    target
)
for caller in all_callers:
    print(caller)


cytoscape_main.py::cytoscape_workflow_by_titles
pyvis_main.py::workflow_by_titles
cytoscape_main.py::cytoscape_workflow_by_tag
cytoscape_main.py::main
pyvis_main.py::main
pyvis_main.py::workflow_largest_component
pyvis_main.py::workflow_by_tag
pyvis_main.py::workflow_by_category
cytoscape_main.py::cytoscape_workflow_largest_component
cytoscape_main.py::cytoscape_workflow_by_category
